# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


####  Run this cell to set up and start your interactive session.


In [2]:
%stop_session

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
There is no current session.


In [5]:
%idle_timeout 30
%glue_version 6.0
%worker_type G.1X
%number_of_workers 5

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 6.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5


In [8]:
%extra_jars s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar, s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar 
%extra_py_files s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-2.1.1.zip

Extra jars to be included:
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar
Extra py files to be included:
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-2.1.1.zip
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar,s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar


In [35]:
#%%configure
#{
#  "--datalake-formats": "iceberg,delta",
#  "--additional-python-modules": "duckdb,pyarrow,shapely",
#  "--conf": {
#    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
#    "spark.kryo.registrator": "com.esri.geoanalytics.KryoRegistrator",
#    "spark.plugins": "com.esri.geoanalytics.Plugin",
#    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension",
#    "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog",
#    "spark.sql.catalog.glue_catalog": "org.apache.iceberg.spark.SparkCatalog",
#    "spark.sql.catalog.glue_catalog.catalog-impl": "org.apache.iceberg.aws.glue.GlueCatalog",
#    "spark.sql.catalog.glue_catalog.warehouse": "s3://pske-prd-customerexperienceadhoc/spatial_analysis/"
#  }}

The following configurations have been updated: {'--datalake-formats': 'iceberg,delta', '--additional-python-modules': 'duckdb,pyarrow,shapely', '--conf': {'spark.serializer': 'org.apache.spark.serializer.KryoSerializer', 'spark.kryo.registrator': 'com.esri.geoanalytics.KryoRegistrator', 'spark.plugins': 'com.esri.geoanalytics.Plugin', 'spark.sql.extensions': 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension', 'spark.sql.catalog.spark_catalog': 'org.apache.spark.sql.delta.catalog.DeltaCatalog', 'spark.sql.catalog.glue_catalog': 'org.apache.iceberg.spark.SparkCatalog', 'spark.sql.catalog.glue_catalog.catalog-impl': 'org.apache.iceberg.aws.glue.GlueCatalog', 'spark.sql.catalog.glue_catalog.warehouse': 's3://pske-prd-customerexperienceadhoc/spatial_analysis/'}}


In [10]:
%%configure
{
 "--conf": "spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin",
}

The following configurations have been updated: {'--conf': 'spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin'}


In [1]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import expr
from awsglue import DynamicFrame
from pyspark.sql.functions import col, to_timestamp
import pyspark.sql.functions as F

# Initialize Spark session
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
# spark = SparkSession.builder.getOrCreate()
spark = SparkSession.builder \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "com.esri.geoanalytics.KryoRegistrator") \
    .config("spark.plugins", "com.esri.geoanalytics.Plugin") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3://pske-prd-datalake/") \
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog") \
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .getOrCreate()
job = Job(glueContext)

# Check active session configs
print("Spark Extensions:", spark.conf.get("spark.sql.extensions", "None"))
print("Serializer:", spark.conf.get("spark.serializer", "None"))
print("Spark Session successfully instantiated!")


Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 30
Session ID: 298d6562-2b3a-447a-9cd5-05321d36e0bf
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--conf spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin
--extra-py-files s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-2.1.1.zip
--extra-jars s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar,s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar
Waiting for session 298d6562-2b3a-447a-9cd5-05321d36e0bf to get into ready status...
Session 298d6562-2b3a-447a-9cd5-05321d36e0bf has been created.
Spark Extensions: com.esri.geoanalytics.sq

In [2]:
import geoanalytics
from geoanalytics.sql import functions as ST
# Authenticate with the license file
geoanalytics.auth(username = "PTL_GAE",password = "MICXA_1008-10!")

In [3]:
# global state Filter
STATE = "DC"
STATE_LOWER = STATE.lower()
print(f"Global state filter set to: {STATE}")

Global state filter set to: DC


In [ ]:
df_polk = glueContext.create_data_frame.from_catalog(
    database="ptl_marketuniverse",
    table_name="mu_polk",
    additional_options={"datalake_formats": "iceberg"}
)
print("Iceberg Polk table loaded successfully via GlueContext.")

df_rigdig = glueContext.create_data_frame.from_catalog(
    database="ptl_marketuniverse",
    table_name="mu_rigdig",
    additional_options={"datalake_formats": "iceberg"}
)
print("Iceberg RigDig table loaded successfully via GlueContext.")

df_mkt_analytics = glueContext.create_data_frame.from_catalog(
    database="ptl_marketuniverse",
    table_name="marketing_analytics",
    additional_options={"datalake_formats": "iceberg"}
)
print("Iceberg marketing_analytics table loaded successfully via GlueContext.")

df_dnb_master = spark.read.table("ptl_marketuniverse.mu_dnb_data_master")
print("Delta D&B Master table loaded successfully via Spark.")

df_sales_force = glueContext.create_data_frame.from_catalog(
    database="ptl_marketuniverse",
    table_name="sales_force_master",
    additional_options={"datalake_formats": "iceberg"}
)
print("Iceberg Sales Force Master table loaded successfully via GlueContext.")

# Register as temp views so downstream non-spatial logic can run as spark.sql
df_polk.createOrReplaceTempView("v_polk")
df_rigdig.createOrReplaceTempView("v_rigdig")
df_mkt_analytics.createOrReplaceTempView("v_mkt_analytics")
df_dnb_master.createOrReplaceTempView("v_dnb_master")
df_sales_force.createOrReplaceTempView("v_sales_force")

df_mkt_analytics.select("final_duns_number").show(1)


In [ ]:
# 1. Non-spatial filter (SQL): keep only D&B rows with usable coordinates
df_dnb_valid = spark.sql("""
    SELECT *
    FROM v_dnb_master
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL
""")

# Spatial: build point geometry (geoanalytics ST function -- not expressible in plain spark.sql)
df_dnb_pts = df_dnb_valid.withColumn("geometry", ST.point("longitude", "latitude", 4326))

# 2. Read GeoJSON boundary from S3
# NOTE: this boundary file lives under an /exports/ folder that's normally
# for pipeline OUTPUT, not reference input -- worth moving it to a dedicated boundaries/
# location so it doesn't get confused with (or overwritten by) an actual export.
s3_geojson_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/exports/dnb_parcel_match_dc_geopqt/DC_district_Bndy.geojson"

df_boundary_raw = spark.read.format("geojson").load(s3_geojson_path)
df_boundary_raw.createOrReplaceTempView("v_boundary_raw")

# Non-spatial filter (SQL): select target district boundary row
df_boundary = spark.sql("""
    SELECT *
    FROM v_boundary_raw
    WHERE district = '0860 WASHINGTON DC'
""")

# df_boundary is a single filtered row, df_dnb_pts is the full DNB master (millions of rows).
# Force broadcast of the small side so the cheap side gets indexed instead of leaving it to
# Spark's cost-based guess (stats are often missing/stale right after a format read + filter) --
# the same class of bug that caused the VA broadcast OOM further down in this notebook.
# 3. Perform Spatial Join using ST.contains (spatial -- kept as DataFrame API)
df_dnb_dc = df_dnb_pts.join(
    F.broadcast(df_boundary),
    ST.contains(df_boundary["geometry"], df_dnb_pts["geometry"]),
    how="inner"
).select(df_dnb_pts["*"])

df_dnb_dc.createOrReplaceTempView("v_dnb_dc")

# 4. Preview DC Filtered Market Universe
df_dnb_dc.select("duns_number", "business_name", "latitude", "longitude").show(5, truncate=False)
print(f"Total D&B Master Records in District 0860 WASHINGTON DC: {df_dnb_dc.count():,}")


In [ ]:
df_boundary.count()

In [ ]:
# All non-spatial: equi-joins on duns_number + CASE-based sales segment classification.
mu_df = spark.sql("""
    SELECT
        -- Company Identification & Linkage
        A.duns_number,
        M.final_duns_number,
        A.business_name,
        A.parent_duns_number,
        A.headquarter_duns_number,
        A.dot_linkage,
        A.global_ultimate_duns_number,
        A.global_ultimate_indicator,
        A.global_ultimate_business_name,
        A.out_of_business_indicator,
        A.duns_linkage,
        A.number_of_family_members,

        -- General Industry & Classification
        A.penske_category,
        A.tradestyle_name,
        A.line_of_business,
        A.formatted_dot_linkage,
        A.us_1987_sic_1,
        A.naics,

        -- Location & Address Fields
        A.street_address,
        A.city_name,
        A.state_province_abbr,
        A.postal_code,
        A.county_name,
        A.latitude,
        A.longitude,

        -- Metrics & Financials
        A.employees_total,
        A.employees_here,
        A.sales_volume_us_dollars,
        A.telephone_number,

        -- Contact Information
        A.chief_exec_officer_full_name,
        A.chief_exec_officer_title,
        A.first_executive_first_name,
        A.first_executive_last_name,
        A.first_executive_title,

        -- Refined 7-Tier Sales Target Segment Classification
        CASE
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 50000000 THEN '1. Mega ($50M+)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 10000000 THEN '2. Large ($10M - $50M)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 5000000  THEN '3. Upper Mid ($5M - $10M)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 2500000  THEN '4. Lower Mid ($2.5M - $5M)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 1000000  THEN '5. Small ($1M - $2.5M)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 250000   THEN '6. Micro ($250K - $1M)'
            WHEN A.sales_volume_us_dollars IS NULL THEN '8. Unknown (No Data)'
            ELSE '7. Nano / Pre-Rev (<$250K)'
        END AS sales_target_segment,

        -- Polk Vehicle Registration Enrichment
        P.confidence_code AS polk_confidence_code,
        P.total_fleet_size_gvw_3_8,

        -- RigDig Enrichment
        R.confidence_code AS rigdig_confidence_code,
        R.ent_usdot_total_pwr,
        R.eqt_class_3to8_units,
        R.eqt_class_all_units,

        -- Sales Force Enrichment (one-to-many on duns_number -- pre-aggregated to one row
        -- per duns_number below, so this join stays one-to-one like the others)
        SF.sf_account_ids,
        SF.sf_confidence_codes
    FROM v_dnb_dc A
    LEFT JOIN v_polk P ON A.duns_number = P.duns_number
    LEFT JOIN v_rigdig R ON A.duns_number = R.duns_number
    LEFT JOIN v_mkt_analytics M ON A.duns_number = M.site_duns_number
    LEFT JOIN (
        SELECT
            duns_number,
            concat_ws(',', collect_list(account_id)) AS sf_account_ids,
            concat_ws(',', collect_list(confidence_code)) AS sf_confidence_codes
        FROM v_sales_force
        GROUP BY duns_number
    ) SF ON A.duns_number = SF.duns_number
""")

mu_df.createOrReplaceTempView("v_mu")

# 5. Filter for target State (DC) to construct dc_mu_df
# dc_mu_df = spark.sql(f"SELECT * FROM v_mu WHERE state_province_abbr = '{STATE}'")

# 6. Execute Counts and Breakdown
dc_mu_df = mu_df
print(f"Total Market Universe Companies ({STATE}): {dc_mu_df.count()}")

print("\nRefined Breakdown by Sales Target Segment:")
spark.sql("""
    SELECT sales_target_segment, COUNT(*) AS count
    FROM v_mu
    GROUP BY sales_target_segment
    ORDER BY sales_target_segment
""").show(truncate=False)


In [ ]:
# apply is_child logic (non-spatial CASE WHEN)
dc_mu_df = spark.sql("""
    SELECT *,
        CASE
            WHEN duns_linkage LIKE '%|%' OR dot_linkage LIKE '%.%' THEN 'Yes'
            ELSE 'No'
        END AS is_child
    FROM v_mu
""")
dc_mu_df.createOrReplaceTempView("v_mu")
print("child defined")


In [ ]:
spark.sql("SELECT COUNT(*) AS cnt FROM v_mu WHERE number_of_family_members <= 1").show()

In [ ]:
# NOTE (no functional change): this file is named LSR_DC_SF_Data.csv but
# dc_ps_join below ends up getting filtered by state_province_abbr for DC, MD, *and* VA in the
# per-state cells further down. If this file is meant to be the single combined DC-metro
# territory extract (covering all three jurisdictions under one "DC territory" label), that's
# fine -- but worth confirming, since if it's actually DC-only, the MD/VA per-state filters
# below will silently return near-zero prospects instead of erroring.
# Read LSR report from Territory analysis
dc_prospects_raw = spark.read.format("csv")\
                            .option("header", "true")\
                            .option("inferSchema", "true")\
                            .load("s3://pske-prd-customerexperienceadhoc/spatial_analysis/lsr_imports/LSR_DC_SF_Data.csv")

dc_prospects_raw = dc_prospects_raw.toDF(*[col.lower() for col in dc_prospects_raw.columns])
dc_prospects_raw.createOrReplaceTempView("v_dc_prospects_raw")

# NOTE: dropping two known columns by name -- Apache Spark SQL (unlike Databricks/BigQuery SQL)
# has no "SELECT * EXCEPT (...)" syntax, and the CSV schema is inferred/dynamic, so a full
# explicit column list isn't available here. `.drop()` is the correct tool for this.
dc_prospects_df = dc_prospects_raw.drop("street_address", "business_name")
dc_prospects_df.createOrReplaceTempView("v_dc_prospects")

print(dc_prospects_df.count())
spark.sql("SELECT sf_account_id, COUNT(*) AS count FROM v_dc_prospects_raw GROUP BY sf_account_id").show(truncate=False)


In [ ]:
# Non-spatial inner join on duns_number.
# JOIN ... USING (duns_number) merges the key into a single output column, matching the
# behavior of the original .join(df, "duns_number", "inner") call (a plain ON join would keep
# both sides' duns_number columns and require disambiguation).
# (Filter out the sf_account_id (sales force account not present): .where(col("sf_account_id") == "No"))
dc_ps_join = spark.sql("""
    SELECT *
    FROM v_mu
    JOIN v_dc_prospects USING (duns_number)
""")
dc_ps_join.createOrReplaceTempView("v_dc_ps_join")
print(f"{STATE} prospect count: {dc_ps_join.count()}")


In [ ]:
spark.sql("SELECT sf_account_id, COUNT(*) AS count FROM v_dc_ps_join GROUP BY sf_account_id").show(truncate=False)
dc_ps_join.printSchema()
dc_ps_join.show(1)


In [ ]:
spark.sql("SELECT state_province_abbr, COUNT(*) AS count FROM v_dc_ps_join GROUP BY state_province_abbr").show(truncate=False)

### Per-state Non-Residential Parcel Spatial Join (DC / MD / VA)

In [ ]:
# Configuration & Paths
S3_TEMP_BASE = "s3://pske-prd-customerexperienceadhoc/Regrid_Parcels/temp_stage_output"
BASE_S3_PATH = "s3://pske-prd-customerexperienceadhoc/Regrid_Parcels/State_Level_US_Parcels"


In [ ]:
# Edit this list to re-run only the state(s) that failed, e.g. STATES_TO_RUN = ["VA"]
STATES_TO_RUN = ["DC", "MD", "VA"]

# Only initialize once -- re-running this cell on purpose (e.g. to reset everything) clears it,
# but re-running the loop cell below after a partial failure should NOT wipe prior successes.
if "state_final_dfs" not in globals():
    state_final_dfs = {}


In [ ]:
def process_state(state_code, folder=None):
    """Run the non-residential parcel + D&B spatial join pipeline for one state/district and
    write single-partition Parquet/CSV/GeoParquet exports. Returns the staged output DataFrame."""
    folder = folder or state_code
    print(f"=== STARTING {state_code} PROCESSING ===")

    # 1. Read State Parcels
    s3_path = f"{BASE_S3_PATH}/{folder}/"
    df_state = spark.read.parquet(s3_path).withColumn("source_state", F.lit(state_code))
    df_state.createOrReplaceTempView(f"v_state_parcels_{state_code.lower()}")

    # 2. Filter Non-Residential Parcels + de-dup (non-spatial: SQL)
    df_state_non_res = spark.sql(f"""
        SELECT *
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY parcelnu_1 ORDER BY parcelnu_1) AS _rn
            FROM v_state_parcels_{state_code.lower()}
            WHERE zoning_typ != 'Residential'
               OR lbcs_activ IS NULL
               OR NOT (
                    CAST(lbcs_activ AS INT) BETWEEN 1000 AND 1999
                    OR lbcs_act_1 ILIKE '%Household%'
               )
        )
        WHERE _rn = 1
    """).drop("_rn")

    # 3. Parcel Geometry Processing (spatial -- kept as DataFrame/ST API)
    parcel_nonres_geom = df_state_non_res.withColumn(
        "polygeom", ST.geom_from_binary("geometry", sr=4326)
    ).select(
        "polygeom", "parcelnu_1", "usecode", "zoning", "zoning_des",
        "zoning_typ", "zoning_sub", "lbcs_activ", "owner", "usedesc"
    )
    parcel_nonres_geom = parcel_nonres_geom.st.set_geometry_field("polygeom")

    # 4. Filter Combined dc_ps_join for Target State (non-spatial: SQL)
    dnb_df_geom = spark.sql(f"SELECT * FROM v_dc_ps_join WHERE state_province_abbr = '{state_code}'")
    # Spatial: build point geometry
    dnb_df_geom = dnb_df_geom.withColumn("pointgeom", ST.point("longitude", "latitude", sr=4326))
    dnb_df_geom = dnb_df_geom.st.set_geometry_field("pointgeom")

    # Broadcast the DNB points (small side, a few thousand rows), not the parcel table -- keeps
    # every state consistent and predictable regardless of how big that state's parcel extract
    # is (broadcasting the parcel side instead is what caused the VA OOM historically).
    # 5. Spatial Join (Using native GeoAnalytics spatial execution)
    dnb_parcel_intersect = parcel_nonres_geom.join(
        F.broadcast(dnb_df_geom),
        ST.contains(parcel_nonres_geom["polygeom"], dnb_df_geom["pointgeom"]),
        how="right"
    )
    dnb_parcel_intersect.createOrReplaceTempView(f"v_dnb_parcel_intersect_{state_code.lower()}")

    # 6. Window Deduplication (non-spatial: SQL)
    # NOTE: Apache Spark SQL has no "SELECT * EXCEPT (...)" projection (that's a Databricks/
    # BigQuery extension), and the parcel/D&B schemas here are read dynamically -- so the two
    # helper columns (parcel_flag, rn) are shed with .drop() after the SQL query instead of
    # being enumerated away.
    no_dup_df = spark.sql(f"""
        SELECT *,
            CASE WHEN parcelnu_1 IS NOT NULL THEN 'Y' ELSE 'N' END AS parcel_flag,
            ROW_NUMBER() OVER (
                PARTITION BY duns_number
                ORDER BY CASE WHEN parcelnu_1 IS NOT NULL THEN 'Y' ELSE 'N' END DESC, lbcs_activ
            ) AS rn
        FROM v_dnb_parcel_intersect_{state_code.lower()}
    """).filter("rn = 1").drop("parcel_flag", "rn")

    # Preserve spatial point geometry as shape column
    no_dup_df = no_dup_df.drop("polygeom").withColumnRenamed("pointgeom", "shape")
    no_dup_df = no_dup_df.st.set_geometry_field("shape")

    # 7. MATERIALIZATION STEP: Write to Parquet Once to Break Lineage
    parquet_path = f"{S3_TEMP_BASE}/parquet/dedup_{state_code}/"
    no_dup_df.write.mode("overwrite").parquet(parquet_path)

    # 8. Read back materialized data for lightweight downstream exports
    staged_df = spark.read.parquet(parquet_path).st.set_geometry_field("shape")

    # CSV Export (single file per district)
    csv_path = f"{S3_TEMP_BASE}/csv/dedup_{state_code}/"
    staged_df.drop("shape").drop("geometry").coalesce(1).write.mode("overwrite") \
        .option("header", "true").option("quoteAll", "true").csv(csv_path)

    # GeoParquet Export (single file per district)
    geoparquet_path = f"{S3_TEMP_BASE}/geoparquet/dedup_{state_code}/"
    staged_df.drop("geometry").coalesce(1).write.format("geoparquet").mode("overwrite").save(geoparquet_path)

    record_count = staged_df.count()
    print(f"=== {state_code} COMPLETED: {record_count:,} records written (Parquet, CSV, & GeoParquet) ===\n")
    return staged_df


In [ ]:
for state_code in STATES_TO_RUN:
    state_final_dfs[state_code] = process_state(state_code)

print(f"Completed states: {list(state_final_dfs.keys())}")


### Combine Per-State Results & Export Tri-State Match Set

In [ ]:
missing = [s for s in ("DC", "MD", "VA") if s not in state_final_dfs]
if missing:
    raise RuntimeError(f"Cannot build combined tri-state export -- missing state(s): {missing}. "
                        f"Set STATES_TO_RUN to those state(s) and re-run the loop cell above first.")

state_final_dfs["DC"].createOrReplaceTempView("v_dc_final")
state_final_dfs["MD"].createOrReplaceTempView("v_md_final")
state_final_dfs["VA"].createOrReplaceTempView("v_va_final")

# Non-spatial: union the three state result sets (SQL)
tristate_df = spark.sql("""
    SELECT * FROM v_dc_final
    UNION ALL
    SELECT * FROM v_md_final
    UNION ALL
    SELECT * FROM v_va_final
""")

geoparquet_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/exports/dnb_parcel_match_dc_geopqt"
tristate_df.drop("zoning_des").coalesce(1).write \
                .format("geoparquet") \
                .mode("overwrite") \
                .option("compression", "snappy") \
                .save(geoparquet_s3_path)
print("successfully Exported (GeoParquet)")

# Write out as a single CSV file with headers
csv_export_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/exports/dnb_parcel_match_dc/"
tristate_df.drop("shape").drop("geometry").coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("emptyValue", "") \
    .save(csv_export_s3_path)
print("csv exported")


In [19]:
geoparquet_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/dnb_parcel_match_dc"
dc_pqt = spark.read.parquet(geoparquet_s3_path)
print(dc_pqt.count())

AnalysisException: [PATH_NOT_FOUND] Path does not exist: s3://pske-prd-customerexperienceadhoc/spatial_analysis/dnb_parcel_match_dc.


In [7]:

df_with_geom = dc_pqt.withColumn("shape", ST.geom_from_binary("shape", sr=4326))                    

df_with_geom.select("shape").printSchema()

root
 |-- shape: geometry (nullable = true)


In [28]:
import requests
import geoanalytics
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

portal_url = "https://gisstgportal.penske.com/portal"
oauth_url = f"{portal_url}/sharing/rest/oauth2/token"

# 1. Get OAuth2 Token using App ID and Secret
payload = {
    "client_id": "Vq99z7jaCI13Wapo",
    "client_secret": "d37ae7cd431c4e90a81edce3edea79a6",
    "grant_type": "client_credentials",
    "expiration": 120,
    "f": "json"
}

res = requests.post(oauth_url, data=payload, verify=False).json()

if "access_token" in res:
    oauth_token = res["access_token"]
    print("OAuth Access Token acquired successfully!")
    
    # 2. Register GIS with the OAuth Access Token
    geoanalytics.register_gis(
        name="myGIS",
        url=portal_url,
        token=oauth_token,
        verify_cert=False
    )
else:
    print("OAuth Error:", res)

ConnectionError: HTTPSConnectionPool(host='gisstgportal.penske.com', port=443): Max retries exceeded with url: /portal/sharing/rest/oauth2/token (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fe89a58b650>: Failed to establish a new connection: [Errno -2] Name or service not known'))


In [25]:
import geoanalytics
#geoanalytics.register_gis("myPortal", "https://gisstgportal.penske.com/portal", username="svc_penske_aws_stg", password="Pen35!DWThj2026")
geoanalytics.register_gis("myGIS1", "https://gisstgserver.penske.com/arcgis", username="svc_penske_aws_stg", password="Pen35!DWThj2026")


IllegalArgumentException: GIS login failed.


In [12]:
myFS="https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/World_Cities/FeatureServer/0"
myFSDataFrame = spark.read.format('feature-service').load(myFS)
group = myFSDataFrame.selectExpr("CNTRY_NAME", "POP").groupBy("CNTRY_NAME").avg("POP")
group.where("avg(POP) > 50000").show()

+-------------+------------------+
|   CNTRY_NAME|          avg(POP)|
+-------------+------------------+
|    Nicaragua|           74500.0|
|     Cameroon|          682800.0|
|        Congo|          315500.0|
|       Israel|          308123.5|
|    Indonesia|1148963.4814814816|
|      Myanmar|          583645.8|
|  South Korea|         2215724.9|
|    Australia|1147753.9333333333|
|French Guiana|           57614.0|
| Burkina Faso|126275.86206896552|
|        Ghana|          374058.1|
|         Togo|          209891.5|
|  The Bahamas|          266100.0|
|       Turkey| 408656.7164179105|
|      Armenia|         1079732.0|
|      Somalia|158411.76470588235|
|       Sweden| 66458.33333333333|
|      Croatia|          459320.5|
|  Afghanistan| 142620.6896551724|
|     Thailand|506188.81944444444|
+-------------+------------------+
only showing top 20 rows


In [9]:
# write to portal layer
geoanalytics.register_gis("myGIS", username="svc_penske_aws_stg", password="Pen35!DWThj2026")
service_name = "dnb_prospects"
portal_url = "https://gisstgserver.penske.com/arcgis" 

df_with_geom.write \
    .format("feature-service") \
    .option("gis", "myGIS") \
    .option("serviceName", service_name) \
    .option("layerName", "layer") \
    .option("tags", "dnb, prospects") \
    .option("description", "D&B Non-Residential Prospects Layer") \
    .mode("overwrite") \
    .save()

IllegalArgumentException: GIS login failed: Unable to generate token. Invalid username or password.


In [29]:
def remove_duplicate_columns(df):
    """Removes duplicate column names from a PySpark DataFrame, keeping the first occurrence."""
    cols = []
    for col_name in df.columns:
        if df.columns.count(col_name) > 1:
            # If column name appears multiple times, keep the first one
            if col_name not in [c for c in cols]:
                cols.append(col_name)
        else:
            cols.append(col_name)
    
    # Select columns by positional index to resolve duplicate references
    return df.select(*[df.schema.names[i] for i, name in enumerate(df.columns) if i == df.columns.index(name)])

parcel_land_usedesc_nodups = remove_duplicate_columns(parcel_land_usedesc)


In [88]:
ATHENA_DB = "ptl_customerexp"
ATHENA_TABLE = "dnb_parcel_intersect_dc"

df_athena = final_joined_df.drop("parcel_geometry")
print(df_athena.count())
spark.sql(f"DROP TABLE IF EXISTS {ATHENA_DB}.{ATHENA_TABLE}")

# 3. Create External Table pointing to S3 location
spark.sql(f"""
    CREATE TABLE {ATHENA_DB}.{ATHENA_TABLE}
    USING parquet
    LOCATION '{S3_OUTPUT_PATH}'    
""")

print(f"Wrote Athena table: {ATHENA_DB}.{ATHENA_TABLE}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {ATHENA_DB}.{ATHENA_TABLE}").show()

4196


In [ ]:
'''Ranking Logic Strategy
When a D&B point hits multiple overlapping parcel polygons, rank them by evaluating four attributes in order:

Commercial/Business  (lbcs_activ): D&B records represent commercial entities. Prioritize non-residential/commercial land uses (e.g., 3000.0 Industrial/Commercial) over residential (1100.0 Household) or NULL codes.
Stacked Parcel Neutrality: If a point sits in a condo stack (ll_stack_u is populated), treat non-stacked base parcels (NULL ll_stack_u) with higher priority unless matching unit-level business activity.
Deterministic Tie-Breaking: Use ll_uuid as a final tie-breaker to prevent non-deterministic partition sorts across distributed Spark tasks.
'''